# 04 — Classificazione Semantica nei Livelli Normativi

Questo notebook classifica ciascun atto normativo del grafo focale in uno dei **5 livelli della piramide normativa** tramite classificazione LLM sul preambolo.

## Framework di classificazione

| Livello | Nome | Contenuto | Segnali linguistici |
|---|---|---|---|
| **G1** | Principi fondamentali | Valori, obiettivi ultimi, diritti fondamentali. Il livello più stabile. | *è garantito*, *costituisce obiettivo*, *è vietato in via assoluta* |
| **G2** | Legge quadro | Traduzione dei principi in regole generali, poteri, soglie, procedure principali. | *il governo può*, *è istituito*, *la soglia è fissata a* |
| **G3** | Regolamenti tecnico-operativi | Dettagli applicativi, moduli, elenchi, codici, tempistiche. Livello delle agenzie e autorità indipendenti. | *il modulo allegato*, *entro X giorni*, *i codici ATECO di cui all'allegato* |
| **G4** | Linee guida e soft law | Interpretazioni, circolari, prassi consolidata. Non vincolante. | *si raccomanda*, *si intende*, *ai fini applicativi* |
| **G5** | Controllo e giurisprudenza | Sentenze, decisioni sanzionatorie, procedure di infrazione. | *il ricorso è accolto*, *si accerta la violazione*, *è irrogata la sanzione* |

## Metodologia

La classificazione avviene in due fasi:

1. **LLM sul preambolo** (per i 1668 nodi con preambolo): il modello riceve il testo del preambolo e le definizioni dei 5 livelli, e restituisce livello + confidenza + motivazione.
2. **Fallback sul titolo** (per i 1098 nodi senza preambolo): se disponibile il titolo, classificazione LLM sul solo titolo.

La **confidenza** (0.0–1.0) è un dato metodologico: un valore basso indica che l'atto è semanticamente ambiguo o ibrido tra livelli — il che è esso stesso un risultato interpretabile come indicatore di qualità normativa.

## Input/Output
- **Input**: `data/output/golden_power/gephi_nodes_focal_preambles.csv` + `gephi_nodes_focal_titled.csv`
- **Output**: `data/output/golden_power/gephi_nodes_focal_classified.csv`

## 0. Setup

In [31]:
from dotenv import load_dotenv
import os
import google.generativeai as genai

genai.configure(api_key=os.environ.get('GEMINI_API_KEY', ''))

In [51]:
import pandas as pd
import json
import os
import sys
import google.generativeai as genai


sys.path.append('..')
from config_golden_power import MATERIA_NAME

output_path      = os.path.join('..', 'data', 'output', MATERIA_NAME)
preambles_file   = os.path.join(output_path, 'gephi_nodes_focal_preambles.csv')
titles_file      = os.path.join(output_path, 'gephi_nodes_focal_titled.csv')
output_file      = os.path.join(output_path, 'gephi_nodes_focal_classified.csv')
checkpoint_file  = os.path.join(output_path, 'classification_checkpoint.csv')

genai.configure(api_key=os.environ.get('GEMINI_API_KEY', ''))
MODEL = 'gemini-2.0-flash'
MAX_TOKENS        = 256   # risposta JSON breve
DELAY_SECONDS     = 5   # pausa tra chiamate API
CHECKPOINT_EVERY  = 50

print(f"Input preamboli: {preambles_file}")
print(f"Input titoli:    {titles_file}")
print(f"Output:          {output_file}")

Input preamboli: ..\data\output\golden_power\gephi_nodes_focal_preambles.csv
Input titoli:    ..\data\output\golden_power\gephi_nodes_focal_titled.csv
Output:          ..\data\output\golden_power\gephi_nodes_focal_classified.csv


## 1. Caricamento e Preparazione Dati

In [33]:
preambles = pd.read_csv(preambles_file)
titles    = pd.read_csv(titles_file)[['Id', 'title']]

# Unisci preamboli e titoli
nodes = preambles.merge(titles, on='Id', how='left')

# Determina il testo da usare per ciascun nodo
# Priorità: preambolo > titolo > niente
nodes['input_text']   = nodes['preamble'].fillna(nodes['title'])
nodes['input_source'] = nodes.apply(
    lambda r: 'preamble' if pd.notna(r['preamble'])
         else 'title'    if pd.notna(r['title'])
         else 'none',
    axis=1
)

print(f"Nodi totali: {len(nodes)}")
print()
print("Fonte del testo per classificazione:")
print(nodes['input_source'].value_counts().to_string())
print()
print(f"Nodi classificabili: {(nodes['input_source'] != 'none').sum()}")
print(f"Nodi non classificabili (nessun testo): {(nodes['input_source'] == 'none').sum()}")

Nodi totali: 4904

Fonte del testo per classificazione:
input_source
preamble    3177
none        1593
title        134

Nodi classificabili: 3311
Nodi non classificabili (nessun testo): 1593


## 2. Caricamento Checkpoint

In [34]:
if os.path.exists(checkpoint_file):
    checkpoint   = pd.read_csv(checkpoint_file)
    already_done = set(checkpoint['Id'])
    print(f"Checkpoint trovato: {len(already_done)} nodi gia classificati")
    print(f"Nodi rimanenti:     {len(nodes) - len(already_done)}")
else:
    checkpoint   = pd.DataFrame(columns=[
        'Id', 'layer', 'confidence', 'motivation', 'input_source', 'classification_status'
    ])
    already_done = set()
    print("Nessun checkpoint trovato, si parte da zero")

nodes_todo = nodes[
    (~nodes['Id'].isin(already_done)) &
    (nodes['input_source'] != 'none')
].copy()

print(f"Da classificare ora: {len(nodes_todo)}")

Nessun checkpoint trovato, si parte da zero
Da classificare ora: 3311


## 3. Prompt e Funzione di Classificazione

Il prompt incorpora le definizioni ufficiali dei livelli normativi (fonte: Comunicazione Commissione 2007 sul framework Lamfalussy, adattato a piramide normativa generica) e richiede una risposta JSON strutturata con livello, confidenza e motivazione.

In [46]:
SYSTEM_PROMPT = """Sei un esperto di diritto dell'Unione Europea specializzato in tecnica legislativa e gerarchia delle fonti normative. Il tuo compito è classificare atti normativi UE secondo il framework Lamfalussy.

Rispondi ESCLUSIVAMENTE con un oggetto JSON valido, senza testo aggiuntivo, senza backtick, senza commenti."""

CLASSIFICATION_PROMPT = """Analizza il seguente atto normativo UE e distribuisci il suo contenuto sui 5 livelli del framework Lamfalussy.

LIVELLI:

G1 - PRINCIPI FONDAMENTALI
Contiene: valori fondanti, obiettivi ultimi dell'ordinamento, diritti e doveri fondamentali. Il livello più stabile e difficile da modificare.
Tipicamente: Trattati, Carta dei diritti fondamentali, principi generali del diritto UE.
Segnali: "is guaranteed", "constitutes a fundamental objective", "is an exclusive competence", "shall be prohibited", "Having regard to the Treaty"

G2 - LEGGE QUADRO
Contiene: traduzione dei principi in regole generali operative. Definisce poteri, soglie, procedure principali, istituzione di organi. Livello del dibattito parlamentare (PE + Consiglio).
Tipicamente: Regolamenti del PE e del Consiglio, Direttive.
Segnali: "the Commission shall", "Member States shall", "is hereby established", "the threshold shall be", "ordinary legislative procedure"

G3 - REGOLAMENTI TECNICO-OPERATIVI
Contiene: dettagli tecnici per rendere operativa la legge quadro. Moduli, elenchi, codici, tempistiche, standard tecnici. Livello delle agenzie e autorità indipendenti.
Tipicamente: Regolamenti delegati, Regolamenti di esecuzione, Decisioni tecniche della Commissione.
Segnali: "pursuant to Article X of Regulation", "the standard form", "the list set out in the Annex", "within X days", "Commission Delegated", "Commission Implementing"

G4 - LINEE GUIDA E SOFT LAW
Contiene: interpretazioni delle regole, prassi applicativa, raccomandazioni non vincolanti.
Tipicamente: Raccomandazioni, Comunicazioni della Commissione, Linee guida di autorità di vigilanza.
Segnali: "recommends", "should be understood as", "for the purposes of applying", "non-binding", "guidelines", "best practice"

G5 - CONTROLLO E GIURISPRUDENZA
Contiene: applicazione forzosa delle regole, sanzioni, interpretazione autentica in caso di conflitto.
Tipicamente: Sentenze CGUE, Decisioni di infrazione, Provvedimenti sanzionatori.
Segnali: "the Court rules", "the action is dismissed", "infringement", "the fine", "judgment", "annuls"

ISTRUZIONI:
- Assegna una percentuale a ciascun livello (G1, G2, G3, G4, G5).
- Le percentuali devono sommare esattamente a 100.
- Un atto ben scritto secondo Lamfalussy dovrebbe concentrarsi su UN solo livello (es. G2: 90%, altri: 0-5% ciascuno).
- Un atto che mescola principi e regole operative avrà percentuali distribuite su più livelli.
- La motivazione deve essere in italiano, massimo 8 parole.

TESTO DELL'ATTO:
{testo}

Rispondi SOLO con questo JSON (percentuali intere che sommano a 100):
{{"G1": 0, "G2": 0, "G3": 0, "G4": 0, "G5": 0, "motivation": "..."}}"""


def classify_act(text, source='preamble'):
    if pd.isna(text) or text == '':
        return None, None, None, None, 'no_text'

    text_truncated = str(text)[:1500]
    prompt = CLASSIFICATION_PROMPT.format(testo=text_truncated)

    try:
        model    = genai.GenerativeModel(
            model_name=MODEL,
            system_instruction=SYSTEM_PROMPT,
        )
        response = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                temperature=0.1,
                max_output_tokens=2048,
                response_mime_type="application/json", 
            ),
        )

        content = response.text.strip() if response.text else ''        

        parsed = json.loads(content)

        layers = ['G1', 'G2', 'G3', 'G4', 'G5']
        dist   = {}
        for l in layers:
            val     = parsed.get(l, 0)
            dist[l] = float(val) if val is not None else 0.0

        total = sum(dist.values())
        if total == 0:
            return None, None, None, None, 'invalid_distribution'

        if abs(total - 100) > 1:
            dist = {l: dist[l] / total * 100 for l in layers}

        dominant_layer = max(dist, key=dist.get)
        purity         = dist[dominant_layer] / 100.0
        motivation     = parsed.get('motivation', '')

        return dist, dominant_layer, purity, motivation, 'ok'

    except json.JSONDecodeError:
        return None, None, None, None, 'json_error'
    except Exception as e:
        return None, None, None, None, f'error: {str(e)[:50]}'


# Test su 32019R0452
print("Test su 32019R0452 (FDI Screening)...")
sample = nodes[nodes['Label'] == '32019R0452']['input_text'].iloc[0]
dist, dominant, purity, motiv, status = classify_act(sample)
print(f"  Status: {status}")

if status == 'ok':
    print(f"  Dominante:     {dominant}  (purezza={purity:.2f})")
    print(f"  Distribuzione: {dist}")
    print(f"  Motiv:         {motiv}")
else:
    print(f"  Classificazione fallita — controlla API key o connessione")

Test su 32019R0452 (FDI Screening)...
  Status: ok
  Dominante:     G2  (purezza=0.60)
  Distribuzione: {'G1': 40.0, 'G2': 60.0, 'G3': 0.0, 'G4': 0.0, 'G5': 0.0}
  Motiv:         Mescola principi fondanti con la base procedurale di una legge quadro.


## 3b. Test su Campione

Prima di classificare tutti i nodi, verifica la qualita' delle classificazioni su un campione stratificato.
Il campione include almeno un nodo per tipo di atto (Regulation, Directive, Decision, Treaty, Case_Law)
e per fonte del testo (preambolo vs titolo).

**Esegui questa cella e controlla manualmente che le classificazioni siano sensate prima di procedere con la cella 4.**

In [52]:
import random

SAMPLE_SIZE = 20 
random.seed(42)

# Campione stratificato: almeno un nodo per tipo e per fonte
sample_parts = []

for legal_type in nodes['LegalType'].dropna().unique():
    for source in ['preamble', 'title']:
        subset = nodes[
            (nodes['LegalType'] == legal_type) &
            (nodes['input_source'] == source) &
            (nodes['input_text'].notna())
        ]
        if len(subset) > 0:
            sample_parts.append(subset.sample(1, random_state=42))

sample = pd.concat(sample_parts).drop_duplicates(subset=['Id'])

# Integra con nodi random se il campione e' troppo piccolo
if len(sample) < SAMPLE_SIZE:
    extra = nodes[
        (~nodes['Id'].isin(sample['Id'])) &
        (nodes['input_text'].notna())
    ].sample(min(SAMPLE_SIZE - len(sample), len(nodes)), random_state=42)
    sample = pd.concat([sample, extra])

print(f'Campione di test: {len(sample)} nodi')
print(f'Per tipo: {sample["LegalType"].value_counts().to_dict()}')
print(f'Per fonte: {sample["input_source"].value_counts().to_dict()}')
print()

# Classifica il campione
sample_results = []

def classify_with_retry(text, source, max_retries=1):
    for attempt in range(max_retries + 1):
        try:
            return classify_act(text, source)
        except Exception as e:
            err_str = str(e)
            if '429' in err_str and attempt < max_retries:
                print(f"    Rate limit, aspetto 60s (tentativo {attempt + 1}/{max_retries})...")
                time.sleep(60)
            else:
                if '429' in err_str:
                    return None, None, None, None, 'rate_limit'
                return None, None, None, None, f'error: {err_str[:50]}'
    return None, None, None, None, 'max_retries_exceeded'

for i, (_, row) in enumerate(sample.iterrows()):
    dist, dominant, purity, motivation, status = classify_with_retry(
        row['input_text'], row['input_source']
    )
    
    sample_results.append({
        'Label':        row['Label'],
        'LegalType':    row['LegalType'],
        'input_source': row['input_source'],
        'dominant_layer': dominant,
        'purity':       purity,
        'G1': dist['G1'] if dist else None,
        'G2': dist['G2'] if dist else None,
        'G3': dist['G3'] if dist else None,
        'G4': dist['G4'] if dist else None,
        'G5': dist['G5'] if dist else None,
        'motivation':   motivation,
        'status':       status,
    })
    
    purity_display = f"{purity:.2f}" if purity is not None else "N/A"
    print(f'[{i+1:>2}/{len(sample)}] {row["Label"]:<20} → {dominant} '
          f'(purezza={purity_display}) [{status}]')
    time.sleep(DELAY_SECONDS)


print()
print('=== RISULTATI CAMPIONE ===')
print('Distribuzione layer dominante:')
print(sample_df['dominant_layer'].value_counts().to_string())
print()
print('Purezza media per layer dominante:')
print(sample_df.groupby('dominant_layer')['purity'].mean().round(2).to_string())
print()
print('Dettaglio (atti più ibridi = purezza bassa = violazione Lamfalussy):')
for _, r in sample_df.sort_values('purity').iterrows():
    bar = '█' * int(r['purity'] * 20) + '░' * (20 - int(r['purity'] * 20))
    print(f"  [{bar}] {r['purity']:.2f} | {r['dominant_layer']} | {r['Label']}")
    print(f"    → {str(r['motivation'])[:100]}")
    print()

print('---')
print('Controlla manualmente che le classificazioni siano corrette.')
print('Se sono sensate, procedi con la cella 4 per classificare tutti i nodi.')
print('Se vedi anomalie, modifica il prompt nella cella 3 e riesegui questo test.')

Campione di test: 20 nodi
Per tipo: {'Decision': 7, 'Regulation': 4, 'Directive': 2, 'Legislative_Act': 2, 'Complementary_Act': 2, 'Recommendation': 1, 'Guidelines': 1, 'Case_Law': 1}
Per fonte: {'preamble': 15, 'title': 5}

[ 1/20] 32022R1369           → None (purezza=N/A) [error: 429 You exceeded your current quota, please check ]
[ 2/20] 32021R0821           → None (purezza=N/A) [error: 429 You exceeded your current quota, please check ]


KeyboardInterrupt: 

## 4. Classificazione con Checkpoint

Con 2.766 nodi e 0.3s di delay il tempo stimato e circa **15 minuti**. Se viene interrotto, riesegui questa cella: ripartira' dal checkpoint automaticamente.

In [ ]:
results = []
total   = len(nodes_todo)
n_ok    = 0
n_err   = 0

print(f"Inizio classificazione: {total} nodi")
print(f"Tempo stimato: ~{total * DELAY_SECONDS / 60:.0f} minuti\n")

for i, (_, row) in enumerate(nodes_todo.iterrows()):
    node_id = row['Id']
    text    = row['input_text']
    source  = row['input_source']

    dist, dominant, purity, motivation, status = classify_act(text, source)

    if status == 'ok':
        n_ok += 1
    else:
        n_err += 1
        if status == 'rate_limit':
            time.sleep(15) 

    results.append({
        'Id':                    node_id,
        'dominant_layer':        dominant,
        'purity':                purity,
        'G1': dist['G1'] if dist else None,
        'G2': dist['G2'] if dist else None,
        'G3': dist['G3'] if dist else None,
        'G4': dist['G4'] if dist else None,
        'G5': dist['G5'] if dist else None,
        'motivation':            motivation,
        'input_source':          source,
        'classification_status': status,
    })

    if (i + 1) % 10 == 0 or (i + 1) == total:
        pct = (i + 1) / total * 100
        print(f"  [{i+1:>4}/{total}] {pct:5.1f}%  ok: {n_ok}  errori: {n_err}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        batch              = pd.DataFrame(results)
        checkpoint_updated = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
        checkpoint_updated.to_csv(checkpoint_file, index=False)
        print(f"  --> Checkpoint salvato ({len(checkpoint_updated)} nodi totali)")

    time.sleep(DELAY_SECONDS)

batch            = pd.DataFrame(results)
checkpoint_final = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
checkpoint_final.to_csv(checkpoint_file, index=False)
print(f"\nClassificazione completata.")
print(f"  OK:     {(checkpoint_final['classification_status'] == 'ok').sum()}")
print(f"  Errori: {(checkpoint_final['classification_status'] != 'ok').sum()}")
print()
print("Distribuzione layer:")
print(checkpoint_final['layer'].value_counts().to_string())

## 5. Export e Calcolo Metriche

In [ ]:
class_df = pd.read_csv(checkpoint_file)[[
    'Id', 'dominant_layer', 'purity',
    'G1', 'G2', 'G3', 'G4', 'G5',
    'motivation', 'input_source', 'classification_status'
]]

class_df = class_df.rename(columns={
    'dominant_layer': 'layer',
    'purity':         'layer_confidence',
})

nodes_classified = nodes.merge(class_df, on='Id', how='left')
nodes_classified.to_csv(output_file, index=False)

print(f"File salvato: {output_file}")
print(f"  Classificati: {nodes_classified['layer'].notna().sum()}")
print(f"\nDistribuzione layer dominante:")
print(nodes_classified['layer'].value_counts().to_string())
print(f"\nPurezza media per layer:")
print(nodes_classified.groupby('layer')['layer_confidence'].mean().round(2).sort_index().to_string())

In [ ]:
import numpy as np

classified = nodes_classified[nodes_classified['layer'].notna()].copy()

LAYERS = ['G1', 'G2', 'G3', 'G4', 'G5']

# Calcola entropia di Shannon sulla distribuzione dei layer
def shannon_entropy(row):
    probs = np.array([row[l] for l in LAYERS], dtype=float) / 100.0
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs)) if len(probs) > 0 else 0.0

classified['layer_entropy'] = classified.apply(shannon_entropy, axis=1)

# Atti ibridi: purezza < soglia
PURITY_THRESHOLD = 0.7
hybrid = classified[classified['layer_confidence'] < PURITY_THRESHOLD]

print("=== ANALISI LAMFALUSSY ===")
print()
print(f"Soglia purezza: {PURITY_THRESHOLD}")
print(f"Atti puri  (purezza >= {PURITY_THRESHOLD}): {len(classified) - len(hybrid)} ({(len(classified)-len(hybrid))/len(classified)*100:.1f}%)")
print(f"Atti ibridi (purezza < {PURITY_THRESHOLD}):  {len(hybrid)} ({len(hybrid)/len(classified)*100:.1f}%)")
print()
print("Purezza media per layer:")
print(classified.groupby('layer')['layer_confidence'].mean().round(2).sort_index().to_string())
print()
print("Entropia media per layer (0=atto puro, 2.32=atto uniformemente distribuito):")
print(classified.groupby('layer')['layer_entropy'].mean().round(2).sort_index().to_string())
print()
print("Co-occurrence rate G2-G3 (mescolanza più critica):")
cooc_g2g3 = ((classified['G2'] > 20) & (classified['G3'] > 20)).sum() / len(classified) * 100
print(f"  Atti con G2>20% E G3>20%: {cooc_g2g3:.1f}%")

# Aggiorna il file con le nuove colonne
nodes_classified['layer_entropy'] = classified['layer_entropy']
nodes_classified.to_csv(output_file, index=False)
print(f"\nFile aggiornato con colonne layer_entropy: {output_file}")

## 6. Analisi Ambiguita'

Gli atti con confidenza bassa (< 0.6) sono i candidati a essere strutturalmente ibridi o normativamente ambigui. Questa e' una delle risposte di ricerca principali: quanti atti sono difficilmente classificabili e dove si concentrano?

In [ ]:
AMBIGUITY_THRESHOLD = 0.6

classified = nodes_classified[nodes_classified['layer'].notna()].copy()
ambiguous  = classified[classified['confidence'] < AMBIGUITY_THRESHOLD]

print(f"=== ANALISI AMBIGUITA' (soglia confidenza < {AMBIGUITY_THRESHOLD}) ===")
print()
print(f"Atti ambigui:      {len(ambiguous)} ({len(ambiguous)/len(classified)*100:.1f}%)")
print(f"Atti non ambigui:  {len(classified) - len(ambiguous)} ({(len(classified)-len(ambiguous))/len(classified)*100:.1f}%)")
print()
print("Atti ambigui per layer assegnato:")
print(ambiguous['layer'].value_counts().to_string())
print()
print("Confidenza media per layer:")
conf_stats = classified.groupby('layer')['confidence'].agg(['mean', 'median', 'min'])
print(conf_stats.round(2).to_string())
print()
print("Esempi di atti ambigui (confidenza piu' bassa):")
esempi = ambiguous.nsmallest(10, 'confidence')[['Label', 'layer', 'confidence', 'motivation']]
for _, r in esempi.iterrows():
    print(f"  {r['layer']} | conf={r['confidence']:.2f} | {r['Label']} | {str(r['motivation'])[:80]}")

## 7. Analisi violazione Lamfalussy

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

LAYERS = ['G1', 'G2', 'G3', 'G4', 'G5']
LAYER_COLORS = {'G1': '#8B5CF6', 'G2': '#10B981', 'G3': '#F59E0B', 'G4': '#E879F9', 'G5': '#EF4444'}

classified = df[df['layer'].notna() & df['G1'].notna()].copy()

# ── Figura 1: scatter purezza vs indegree ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

colors = [LAYER_COLORS.get(l, '#999') for l in classified['layer']]
scatter = ax.scatter(
    classified['layer_confidence'],
    classified['indegree'],
    c=colors, alpha=0.6, s=30, edgecolors='none'
)

ax.axvline(x=0.7, color='red', linestyle='--', linewidth=1.2, label='Soglia purezza (0.7)')
ax.set_xlabel('Purezza Lamfalussy (1 = atto perfettamente classificabile)', fontsize=11)
ax.set_ylabel('Indegree (numero di citazioni ricevute)', fontsize=11)
ax.set_title('Purezza Lamfalussy vs Centralità nella Rete', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.05)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=LAYER_COLORS[l], label=l) for l in LAYER_ORDER]
legend_elements.append(plt.Line2D([0], [0], color='red', linestyle='--', label='Soglia purezza'))
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGS, 'fig_purity_vs_indegree.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: fig_purity_vs_indegree.png")

## 8. Stacked bar: distribuzione layer per i top 20 atti più citati

In [ ]:
top20 = classified.nlargest(20, 'indegree').copy()

fig, ax = plt.subplots(figsize=(13, 7))

y_pos   = range(len(top20))
bottoms = np.zeros(len(top20))

for layer in LAYERS:
    values = top20[layer].values
    bars   = ax.barh(list(y_pos), values, left=bottoms,
                     color=LAYER_COLORS[layer], alpha=0.85,
                     label=layer, edgecolor='white', linewidth=0.5)
    bottoms += values

# Etichetta con CELEX + indegree
labels = [f"{row['Label']}  (in={row['indegree']})" for _, row in top20.iterrows()]
ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, fontsize=8.5)
ax.invert_yaxis()
ax.set_xlabel('Distribuzione % sui layer Lamfalussy', fontsize=11)
ax.set_title('Composizione Lamfalussy — Top 20 atti per indegree\n'
             '(una barra uniforme = atto ben scritto; barra mista = violazione Lamfalussy)',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 100)

# Linea verticale al 70% come riferimento purezza
ax.axvline(x=70, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Soglia purezza 70%')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGS, 'fig_lamfalussy_top20.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: fig_lamfalussy_top20.png")

## 9. Heatmap entropia per LegalType

In [ ]:
pivot_entropy = classified.groupby(['layer', 'LegalType'])['layer_entropy'].mean().unstack()
pivot_entropy = pivot_entropy.reindex(LAYER_ORDER)

# Tieni solo i tipi con almeno 5 atti
valid_types = classified['LegalType'].value_counts()
valid_types = valid_types[valid_types >= 5].index
pivot_entropy = pivot_entropy[[c for c in valid_types if c in pivot_entropy.columns]]

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot_entropy.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=2.32)

ax.set_xticks(range(len(pivot_entropy.columns)))
ax.set_xticklabels(pivot_entropy.columns, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(len(LAYER_ORDER)))
ax.set_yticklabels(LAYER_ORDER, fontsize=10)

# Valori nelle celle
for i in range(pivot_entropy.shape[0]):
    for j in range(pivot_entropy.shape[1]):
        val = pivot_entropy.iloc[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color='black')

plt.colorbar(im, ax=ax, label='Entropia media (0=puro, 2.32=uniforme)')
ax.set_title('Entropia Lamfalussy per Layer e Tipo di Atto\n'
             '(rosso = atti che mescolano più livelli = violazione Lamfalussy)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGS, 'fig_entropy_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: fig_entropy_heatmap.png")